In [ ]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms

import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

os.chdir('/content/drive/MyDrive/Project/BrainRegionId/Project43/Code')
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

In [ ]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-cb07b741-2923-71ab-8712-270b851e14b3)


In [ ]:
def net_train_AnyNet_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                if epoch0 == 0 and epoch == 0:
                    if accu_fun(py_train, y_train) == (torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu():
                        print('accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                    else:
                        print('accu_fun is wrong>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                        return
                print(accu_fun(py_train, y_train))
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append(accu_fun(py_train, y_train))
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))

        Classifier.eval()
        acu_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append(accu_fun(py_valid, y_valid))
            # acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/VISp/AnyNet_L_Allen_{key}{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/VISp/AnyNet_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))

    torch.save(acu_array_train, train_args['save_dir'] + f'/Model/VISp/AnyNet_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(acu_array_valid, train_args['save_dir'] + f'/Model/VISp/AnyNet_L_Allen_{key}_valid_acu{ind}.pt')

    return


def net_train_ViT_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))

        Classifier.eval()
        acu_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/VISp/ViT_L_Allen_{key}{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/VISp/ViT_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/VISp/ViT_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/VISp/ViT_L_Allen_{key}_valid_acu{ind}.pt')

    return

def net_train_RNN_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))

        Classifier.eval()
        acu_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/VISp/RNN_L_Allen_{key}{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/VISp/RNN_L_Allen_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/VISp/RNN_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/VISp/RNN_L_Allen_{key}_valid_acu{ind}.pt')

    return

def net_train_LC_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).flatten(start_dim=1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))

        Classifier.eval()
        acu_valid = []
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).squeeze(1).flatten(start_dim=1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        # if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
        if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
            torch.save(Classifier, train_args['save_dir'] + f'/Model/VISp/LC_L_Allen_{key}{ind}.pth')
            torch.save(epoch, train_args['save_dir'] + f'/Model/VISp/LC_L_Allen_{key}_epoch{ind}.pt')
            acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))


    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/VISp/LC_L_Allen_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/VISp/LC_L_Allen_{key}_valid_acu{ind}.pt')

    return


In [ ]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [ ]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load(file_dir + '/brain_signal_lfp1.pt')
for file_id in [2, 3, 4, 5]:
    brain_signal_lfp = torch.concat([brain_signal_lfp, torch.load(file_dir + f'/brain_signal_lfp{file_id}.pt')], dim=0)
    print(f'Load file id: {file_id}')

# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/brain_signal_lfp')
list_dict = torch.load(file_dir + '/list_dict.pt')
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']

Load file id: 2
Load file id: 3
Load file id: 4
Load file id: 5


In [ ]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [ ]:
def subject_od_ind_gen(list_dict, select_acronym, sub_num):
    intersect_subject = np.unique(np.array(list_dict['subject_list'])[np.argwhere(np.array(list_dict['brain_region_atlas']) == select_acronym[0]).flatten()])
    for acronym in select_acronym[1:]:
        intersect_subject = np.intersect1d(intersect_subject, np.unique(np.array(list_dict['subject_list'])[np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten()]))

    subject_od_list = []
    for acronym in select_acronym:
        subject_acronym = np.unique(np.array(list_dict['subject_list'])[np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten()])
        subject_od_list.append(np.setdiff1d(subject_acronym, intersect_subject)[0 : sub_num])

    subject_od_list = np.unique(np.array(subject_od_list))

    subject_od_ind = []
    for subject in subject_od_list:
        subject_od_ind.append(np.argwhere(np.array(list_dict['subject_list']) == subject).flatten())
    subject_od_ind = np.concatenate(subject_od_ind, axis=0)

    return subject_od_ind, subject_od_list

def dat_ind_gen(list_dict, select_acronym, subject_od_ind, key):

    acronym_select_index = []
    for acronym in select_acronym:
        acronym_select_index.append(np.argwhere(np.array(list_dict['brain_region_atlas']) == acronym).flatten())

    acronym_select_index = np.concatenate(acronym_select_index)


    train_ind = np.setdiff1d(np.intersect1d(np.argwhere(np.array(list_dict['train_list_intest']) == 1).flatten(),
                            np.argwhere(np.array(list_dict['key_list']) == key).flatten()), subject_od_ind)

    valid_ind = np.setdiff1d(np.intersect1d(np.argwhere(np.array(list_dict['valid_list_intest']) == 1).flatten(),
                                np.argwhere(np.array(list_dict['key_list']) == key).flatten()), subject_od_ind)

    test_ind = np.setdiff1d(np.intersect1d(np.argwhere(np.array(list_dict['test_list_intest']) == 1).flatten(),
                                np.argwhere(np.array(list_dict['key_list']) == key).flatten()), subject_od_ind)

    test_subject_ind = np.intersect1d(subject_od_ind, np.argwhere(np.array(list_dict['key_list']) == key).flatten())

    if len(train_ind) + len(valid_ind) + len(test_ind) + len(test_subject_ind) == len(list_dict['train_list_intest']) // 2:
        print('Matched')

    train_ind = np.intersect1d(train_ind, acronym_select_index)
    valid_ind = np.intersect1d(valid_ind, acronym_select_index)
    test_ind = np.intersect1d(test_ind, acronym_select_index)
    test_subject_ind = np.intersect1d(test_subject_ind, acronym_select_index)

    return train_ind, valid_ind, test_ind, test_subject_ind

In [ ]:
key = 'stimOff_times'
sub_num = 2
select_acronym = ['VISp1', 'VISp2/3', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']
acronym_index_st = np.argwhere(np.array(acronym_list) == select_acronym[0]).flatten()
subject_od_ind, subject_od_list = subject_od_ind_gen(list_dict, select_acronym, sub_num)

train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, select_acronym, subject_od_ind, key)
data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index[train_ind] - acronym_index_st, coordinate_list[train_ind])
data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index[valid_ind] - acronym_index_st, coordinate_list[valid_ind])
data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index[test_ind] - acronym_index_st, coordinate_list[test_ind])

train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

Matched


In [ ]:
c0 = 64 * 4
k0 = 1.0

model_args = {
    'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
    'stem_channels':c0,
}
train_args = {
    'overfitting_thres':0.60,
    'lr':5e-4,
    'norm':True,
    'temp':[True, True],
    'epochs':50,
    'save_dir':'/content/drive/MyDrive/Project/BrainRegionId/Project43',
}

torch.save(subject_od_ind, train_args['save_dir'] + f'/Model/VISp/subject_od_ind_Allen_{key}{0}.pt')
torch.save(subject_od_list, train_args['save_dir'] + f'/Model/VISp/subject_od_list_Allen_{key}{0}.pt')

for ind in range(0, 1):
    Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(select_acronym)).to(device)
    Classifier.apply(init_cnn)
    net_train_AnyNet_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>
tensor(0.2422)
Acu Train: 0.297471821308136
Acu valid: 0.3036110997200012
tensor(0.3594)
Acu Train: 0.3335626423358917
Acu valid: 0.3382638990879059
tensor(0.3125)
Acu Train: 0.36274453997612
Acu valid: 0.29267361760139465
tensor(0.3594)
Acu Train: 0.3802221417427063
Acu valid: 0.3326736092567444
tensor(0.4141)
Acu Train: 0.4116627871990204
Acu valid: 0.3377777934074402
tensor(0.3047)
Acu Train: 0.4142904281616211
Acu valid: 0.3014236092567444
tensor(0.3203)
Acu Train: 0.43556448817253113
Acu valid: 0.3483680784702301
tensor(0.3594)
Acu Train: 0.4550231993198395
Acu valid: 0.33454859256744385
tensor(0.5156)
Acu Train: 0.4690484404563904
Acu valid: 0.4386458694934845
tensor(0.4766)
Acu Train: 0.4980727732181549
Acu valid: 0.4168750047683716
tensor(0.4844)
Acu Train: 0.5228116512298584
Acu valid: 0.39086806774139404
tensor(0.4844)
Acu Train: 0.5437168478965759
Acu valid: 0.38052085041999817
tensor(0.6094)
Acu Train: 0.5749585628509521
Acu v

In [ ]:
train_args['epochs'] = 200
for ind in range(0, 1):
    img_size, patch_size = (224, 28), (28, 28)
    num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
    emb_dropout, blk_dropout = 0.1, 0.1
    Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(select_acronym)).to(device)
    net_train_ViT_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.1484)
Acu Train: 0.24624918401241302
Acu valid: 0.26701390743255615
tensor(0.3359)
Acu Train: 0.2556241750717163
Acu valid: 0.2473958432674408
tensor(0.2266)
Acu Train: 0.2694711685180664
Acu valid: 0.2547916769981384
tensor(0.2500)
Acu Train: 0.28641411662101746
Acu valid: 0.2989236116409302
tensor(0.3125)
Acu Train: 0.3044305145740509
Acu valid: 0.26878470182418823
tensor(0.2734)
Acu Train: 0.3060925304889679
Acu valid: 0.29826390743255615
tensor(0.2969)
Acu Train: 0.31385114789009094
Acu valid: 0.2976388931274414
tensor(0.2734)
Acu Train: 0.31863394379615784
Acu valid: 0.3239583671092987
tensor(0.3750)
Acu Train: 0.33828333020210266
Acu valid: 0.33020836114883423
tensor(0.3438)
Acu Train: 0.3523748517036438
Acu valid: 0.354930579662323
tensor(0.3359)
Acu Train: 0.35813161730766296
Acu valid: 0.35583335161209106
tensor(0.2891)
Acu Train: 0.3618451654911041
Acu valid: 0.366527795791626
tensor(0.3672)
Acu Train: 0.36346566677093506
Acu valid: 0.37034720182418823
tensor(0.3750)

In [ ]:
train_args['epochs'] = 50
RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(select_acronym),
    'data_len':28,
}
for ind in range(0, 1):
    Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
    net_train_RNN_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

tensor(0.1719)
Acu Train: 0.2633289098739624
Acu valid: 0.26510417461395264
tensor(0.2031)
Acu Train: 0.28017657995224
Acu valid: 0.26923611760139465
tensor(0.2812)
Acu Train: 0.305984765291214
Acu valid: 0.3076736032962799
tensor(0.3438)
Acu Train: 0.3205777406692505
Acu valid: 0.32173609733581543
tensor(0.3594)
Acu Train: 0.3353116810321808
Acu valid: 0.3639930784702301
tensor(0.3984)
Acu Train: 0.36924731731414795
Acu valid: 0.36520835757255554
tensor(0.3672)
Acu Train: 0.3894811272621155
Acu valid: 0.3789583444595337
tensor(0.4297)
Acu Train: 0.4099884033203125
Acu valid: 0.4086458683013916
tensor(0.4844)
Acu Train: 0.42818304896354675
Acu valid: 0.3987152874469757
tensor(0.4297)
Acu Train: 0.45109832286834717
Acu valid: 0.3986110985279083
tensor(0.4766)
Acu Train: 0.4755595326423645
Acu valid: 0.4224305748939514
tensor(0.5547)
Acu Train: 0.5156581401824951
Acu valid: 0.4346180856227875
tensor(0.5078)
Acu Train: 0.5451217889785767
Acu valid: 0.40812501311302185
tensor(0.6875)
Acu T

In [ ]:
LC_args = {
    'category_num':len(select_acronym),
}
Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)
# for x_train1, y_train, coordinate_train in train_iter:
#     x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).flatten(start_dim=1)
#     y_train = y_train.to(device)
#     py_train = Classifier(x_train.to(device))
torch.save(Classifier, train_args['save_dir'] + f'/Model/VISp/LC_L_Allen_chance_{key}0.pth')

In [ ]:
train_args['epochs'] = 50
for ind in range(0, 1):
    Classifier = LinearClassifier(spectro_args['spectro_img'][0], LC_args).to(device)
    net_train_LC_L(train_iter, valid_iter, Classifier.to(device), spectro_args, train_args, key, ind, device)

In [ ]:
from google.colab import runtime
runtime.unassign()